In [1]:
import pandas as pd
import numpy as np
import pickle
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, confusion_matrix, classification_report)
from IPython.display import display, Markdown

X_train = pd.read_pickle('../data/processed/tree_ready/X_train.pkl')
X_test = pd.read_pickle('../data/processed/tree_ready/X_test.pkl')
y_train = pd.read_pickle('../data/processed/tree_ready/y_train.pkl')
y_test = pd.read_pickle('../data/processed/tree_ready/y_test.pkl')

results = []
interpretation_log = []  # collects text for saving

def evaluate_and_interpret(name, model, X_test, y_test):
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1] if hasattr(model, 'predict_proba') else None

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_proba) if y_proba is not None else np.nan
    cm = confusion_matrix(y_test, y_pred)
    tn, fp, fn, tp = cm.ravel()

    metrics = {'model': name, 'accuracy': acc, 'precision': prec,
               'recall': rec, 'f1': f1, 'auc': auc}
    results.append(metrics)

    # --- Auto-generated plain-language interpretation ---
    recall_quality = "strong" if rec > 0.7 else "moderate" if rec > 0.5 else "weak"
    auc_quality = "strong" if auc > 0.8 else "moderate" if auc > 0.65 else "weak"

    md = f"""### {name}

**Metrics:**
- Accuracy: {acc:.3f}
- Precision: {prec:.3f}
- Recall (Sensitivity): {rec:.3f}
- F1 Score: {f1:.3f}
- AUC: {auc:.3f}

**Confusion Matrix:** TN={tn}, FP={fp}, FN={fn}, TP={tp}

**Interpretation:**
- The model correctly identifies {rec*100:.1f}% of truly anemic women (recall) — this is {recall_quality} for a health screening context, where missing an anemic case (false negative) is costly.
- Of women predicted anemic, {prec*100:.1f}% actually are (precision).
- AUC of {auc:.3f} indicates {auc_quality} ability to distinguish anemic from non-anemic women overall.
- False negatives (missed anemia cases): {fn} — {"a concerning number given clinical implications" if fn > tp*0.3 else "a relatively acceptable number, though always worth minimizing further"}.
"""
    display(Markdown(md))
    interpretation_log.append(md)
    return metrics

In [2]:
X_train_scaled = pd.read_pickle('../data/processed/neural_ready/X_train.pkl')
X_test_scaled = pd.read_pickle('../data/processed/neural_ready/X_test.pkl')

log_reg = LogisticRegression(class_weight='balanced', max_iter=5000, solver='saga', random_state=42)
log_reg.fit(X_train_scaled, y_train)
evaluate_and_interpret('Logistic Regression', log_reg, X_test_scaled, y_test)

### Logistic Regression

**Metrics:**
- Accuracy: 0.673
- Precision: 0.484
- Recall (Sensitivity): 0.568
- F1 Score: 0.523
- AUC: 0.699

**Confusion Matrix:** TN=333, FP=129, FN=92, TP=121

**Interpretation:**
- The model correctly identifies 56.8% of truly anemic women (recall) — this is moderate for a health screening context, where missing an anemic case (false negative) is costly.
- Of women predicted anemic, 48.4% actually are (precision).
- AUC of 0.699 indicates moderate ability to distinguish anemic from non-anemic women overall.
- False negatives (missed anemia cases): 92 — a concerning number given clinical implications.


{'model': 'Logistic Regression',
 'accuracy': 0.6725925925925926,
 'precision': 0.484,
 'recall': 0.568075117370892,
 'f1': 0.5226781857451404,
 'auc': 0.6986464239985367}

In [3]:
dt = DecisionTreeClassifier(class_weight='balanced', max_depth=6, random_state=42)
dt.fit(X_train, y_train)
evaluate_and_interpret('Decision Tree', dt, X_test, y_test)

### Decision Tree

**Metrics:**
- Accuracy: 0.704
- Precision: 0.531
- Recall (Sensitivity): 0.516
- F1 Score: 0.524
- AUC: 0.700

**Confusion Matrix:** TN=365, FP=97, FN=103, TP=110

**Interpretation:**
- The model correctly identifies 51.6% of truly anemic women (recall) — this is moderate for a health screening context, where missing an anemic case (false negative) is costly.
- Of women predicted anemic, 53.1% actually are (precision).
- AUC of 0.700 indicates moderate ability to distinguish anemic from non-anemic women overall.
- False negatives (missed anemia cases): 103 — a concerning number given clinical implications.


{'model': 'Decision Tree',
 'accuracy': 0.7037037037037037,
 'precision': 0.5314009661835749,
 'recall': 0.5164319248826291,
 'f1': 0.5238095238095238,
 'auc': 0.6997032701258054}

In [4]:
knn = KNeighborsClassifier(n_neighbors=15)
knn.fit(X_train_scaled, y_train)
evaluate_and_interpret('KNN', knn, X_test_scaled, y_test)   # renamed from 'KNN (scaled)'

### KNN

**Metrics:**
- Accuracy: 0.708
- Precision: 0.568
- Recall (Sensitivity): 0.315
- F1 Score: 0.405
- AUC: 0.669

**Confusion Matrix:** TN=411, FP=51, FN=146, TP=67

**Interpretation:**
- The model correctly identifies 31.5% of truly anemic women (recall) — this is weak for a health screening context, where missing an anemic case (false negative) is costly.
- Of women predicted anemic, 56.8% actually are (precision).
- AUC of 0.669 indicates moderate ability to distinguish anemic from non-anemic women overall.
- False negatives (missed anemia cases): 146 — a concerning number given clinical implications.


{'model': 'KNN',
 'accuracy': 0.7081481481481482,
 'precision': 0.5677966101694916,
 'recall': 0.3145539906103286,
 'f1': 0.40483383685800606,
 'auc': 0.6694256447777575}

In [5]:
# Save all interpretations as one .txt file
output_path = '../data/processed/baseline_model_interpretation.txt'
with open(output_path, 'w', encoding='utf-8') as f:
    f.write('\n\n---\n\n'.join(interpretation_log))
print(f"Saved interpretations to {output_path}")

# Save the metrics table too, for the model comparison notebook later
results_df = pd.DataFrame(results)
results_df.to_csv('../results/metrics/baseline_metrics.csv', index=False)
print(results_df)

Saved interpretations to ../data/processed/baseline_model_interpretation.txt
                 model  accuracy  precision    recall        f1       auc
0  Logistic Regression  0.672593   0.484000  0.568075  0.522678  0.698646
1        Decision Tree  0.703704   0.531401  0.516432  0.523810  0.699703
2                  KNN  0.708148   0.567797  0.314554  0.404834  0.669426


In [6]:
knn = KNeighborsClassifier(n_neighbors=15)
knn.fit(X_train_scaled, y_train)
evaluate_and_interpret('KNN (scaled)', knn, X_test_scaled, y_test)

### KNN (scaled)

**Metrics:**
- Accuracy: 0.708
- Precision: 0.568
- Recall (Sensitivity): 0.315
- F1 Score: 0.405
- AUC: 0.669

**Confusion Matrix:** TN=411, FP=51, FN=146, TP=67

**Interpretation:**
- The model correctly identifies 31.5% of truly anemic women (recall) — this is weak for a health screening context, where missing an anemic case (false negative) is costly.
- Of women predicted anemic, 56.8% actually are (precision).
- AUC of 0.669 indicates moderate ability to distinguish anemic from non-anemic women overall.
- False negatives (missed anemia cases): 146 — a concerning number given clinical implications.


{'model': 'KNN (scaled)',
 'accuracy': 0.7081481481481482,
 'precision': 0.5677966101694916,
 'recall': 0.3145539906103286,
 'f1': 0.40483383685800606,
 'auc': 0.6694256447777575}

In [7]:
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression

# SVM — scale-sensitive (like KNN/LR), use neural_ready data
svm = SVC(kernel='rbf', class_weight='balanced', probability=True, random_state=42)
svm.fit(X_train_scaled, y_train)
evaluate_and_interpret('SVM', svm, X_test_scaled, y_test)

# Regularized Logistic Regression (L1/Lasso) — also scale-sensitive
log_reg_l1 = LogisticRegression(penalty='l1', solver='liblinear', class_weight='balanced',
                                  C=0.5, random_state=42)
log_reg_l1.fit(X_train_scaled, y_train)
evaluate_and_interpret('Logistic Regression (L1 Regularized)', log_reg_l1, X_test_scaled, y_test)

### SVM

**Metrics:**
- Accuracy: 0.676
- Precision: 0.488
- Recall (Sensitivity): 0.573
- F1 Score: 0.527
- AUC: 0.693

**Confusion Matrix:** TN=334, FP=128, FN=91, TP=122

**Interpretation:**
- The model correctly identifies 57.3% of truly anemic women (recall) — this is moderate for a health screening context, where missing an anemic case (false negative) is costly.
- Of women predicted anemic, 48.8% actually are (precision).
- AUC of 0.693 indicates moderate ability to distinguish anemic from non-anemic women overall.
- False negatives (missed anemia cases): 91 — a concerning number given clinical implications.


### Logistic Regression (L1 Regularized)

**Metrics:**
- Accuracy: 0.673
- Precision: 0.484
- Recall (Sensitivity): 0.568
- F1 Score: 0.523
- AUC: 0.699

**Confusion Matrix:** TN=333, FP=129, FN=92, TP=121

**Interpretation:**
- The model correctly identifies 56.8% of truly anemic women (recall) — this is moderate for a health screening context, where missing an anemic case (false negative) is costly.
- Of women predicted anemic, 48.4% actually are (precision).
- AUC of 0.699 indicates moderate ability to distinguish anemic from non-anemic women overall.
- False negatives (missed anemia cases): 92 — a concerning number given clinical implications.


{'model': 'Logistic Regression (L1 Regularized)',
 'accuracy': 0.6725925925925926,
 'precision': 0.484,
 'recall': 0.568075117370892,
 'f1': 0.5226781857451404,
 'auc': 0.6994695445399671}

In [8]:
# Save all interpretations
output_path = '../data/processed/baseline_model_interpretation.txt'
with open(output_path, 'w', encoding='utf-8') as f:
    f.write('\n\n---\n\n'.join(interpretation_log))
print(f"Saved interpretations to {output_path}")

results_df = pd.DataFrame(results)
results_df.to_csv('../results/metrics/baseline_metrics.csv', index=False)
print(results_df)

Saved interpretations to ../data/processed/baseline_model_interpretation.txt
                                  model  accuracy  precision    recall  \
0                   Logistic Regression  0.672593   0.484000  0.568075   
1                         Decision Tree  0.703704   0.531401  0.516432   
2                                   KNN  0.708148   0.567797  0.314554   
3                          KNN (scaled)  0.708148   0.567797  0.314554   
4                                   SVM  0.675556   0.488000  0.572770   
5  Logistic Regression (L1 Regularized)  0.672593   0.484000  0.568075   

         f1       auc  
0  0.522678  0.698646  
1  0.523810  0.699703  
2  0.404834  0.669426  
3  0.404834  0.669426  
4  0.526998  0.692758  
5  0.522678  0.699470  


In [9]:
# Save trained baseline models for later reuse
with open('../models/ml/logistic_regression.pkl', 'wb') as f:
    pickle.dump(log_reg, f)
with open('../models/ml/decision_tree.pkl', 'wb') as f:
    pickle.dump(dt, f)
with open('../models/ml/knn.pkl', 'wb') as f:
    pickle.dump(knn, f)
with open('../models/ml/svm.pkl', 'wb') as f:
    pickle.dump(svm, f)